# 1、 创建提示词模板

In [22]:
import os

from dotenv import load_dotenv

SQL_SYSTEM_PROMPT = """
你是一位专业的 MySQL 数据库专家。你的任务是根据用户的需求编写高效、准确的 SQL 查询语句。

工作流要求：
1. 必须首先调用 get_db_schema 工具获取表结构。
2. 根据表结构编写 SQL，并调用 run_sql_query 工具进行验证。
3. 如果工具返回错误，必须根据错误信息进行分析并重新编写 SQL，直到查询成功。
4. 最终只输出分析过程和确认无误的 SQL 语句。

安全约束：
- 禁止执行任何修改数据的操作（UPDATE, DELETE, DROP, INSERT 等）。
- 仅处理与数据库查询相关的请求。
"""

# 2、创建工具

In [23]:
import mysql.connector
from langchain.tools import tool

# 加载变量
load_dotenv()

# 数据库连接配置
DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": os.getenv("MYSQL_PASSWORD"),
    "database": os.getenv("MYSQL_DATABASE"),
    "auth_plugin": "mysql_native_password"
}

@tool
def get_db_schema() -> str:
    """获取数据库中所有表的 DDL 语句，用于了解表结构。"""
    try:
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor()
        cursor.execute("SHOW TABLES")
        tables = [t[0] for t in cursor.fetchall()]

        schemas = []
        for table in tables:
            cursor.execute(f"SHOW CREATE TABLE {table}")
            res = cursor.fetchone()
            if res:
                schemas.append(res[1])

        cursor.close()
        conn.close()
        return "\n\n".join(schemas)
    except Exception as e:
        return f"获取 Schema 失败: {str(e)}"

@tool
def run_sql_query(query: str) -> str:
    """执行生成的 SELECT 语句并返回前 5 条结果。仅限 SELECT 语句。"""
    query_clean = query.strip().lower()
    if not query_clean.startswith("select"):
        return "错误：仅支持 SELECT 查询操作。"

    try:
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor(dictionary=True)
        cursor.execute(query)
        results = cursor.fetchall()
        cursor.close()
        conn.close()
        # 返回前5行结果用于验证
        return f"查询结果(前5行): {str(results[:5])}"
    except Exception as e:
        return f"SQL 执行报错: {str(e)}，请根据报错修正 SQL。"


# 3、配置模型

In [24]:
# 使用指定provider参数
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv

# 获取API密钥
load_dotenv()

ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_BASE_URL = os.getenv("ZHIPUAI_BASE_URL")

model = init_chat_model(
    model="glm-4.5-air",
    model_provider="openai",  # 因为智谱AI兼容OpenAI API
    temperature=0.5,
    timeout=10,
    max_tokens=1000,
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_BASE_URL
)

# 4、定义响应格式

In [25]:
from dataclasses import dataclass
@dataclass
class SQLResponse:
    """SQL Agent 响应结构。"""
    logic_explanation: str  # 查询逻辑说明
    sql_query: str          # 最终生成的 SQL 语句
    is_verified: bool       # 是否通过工具验证成功

# 5、添加内存

In [26]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

# 6、 创建并运行agent

In [27]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

# 创建 Agent 实例
sql_agent = create_agent(
    model=model,
    system_prompt=SQL_SYSTEM_PROMPT,
    tools=[get_db_schema, run_sql_query],
    response_format=ToolStrategy(SQLResponse),
    checkpointer=checkpointer
)

# 调用示例
config = {"configurable": {"thread_id": "sql_session_01"}}
result = sql_agent.invoke(
    {"messages": [{"role": "user", "content": "统计去年每个月的订单总额"}]},
    config=config
)

print(result['structured_response'])

SQLResponse(logic_explanation="根据数据库表结构分析，订单数据存储在 t5_order 表中，包含 order_time（订单时间）和 order_amt（订单金额）字段。要统计去年每个月的订单总额，需要：\n\n1. 确定去年是2023年（因为当前数据是2024年的）\n2. 使用 YEAR(order_time) = 2023 来筛选去年的订单\n3. 按 DATE_FORMAT(order_time, '%Y-%m') 分组统计每个月的订单总额\n4. 按月份排序显示结果\n\n查询结果显示2023年1月有订单数据，但只有一条记录显示，可能是因为2023年其他月份没有订单数据。", sql_query="SELECT \n    DATE_FORMAT(order_time, '%Y-%m') as month,\n    SUM(order_amt) as total_amount\nFROM t5_order\nWHERE YEAR(order_time) = 2023\nGROUP BY DATE_FORMAT(order_time, '%Y-%m')\nORDER BY month;", is_verified=True)
